# Python 101
## Part X. - Vote results scraping
---

<img src="http://www.london24.com/polopoly_fs/1.3024317.1385128334!/image/4183113330.jpg_gen/derivatives/landscape_630/4183113330.jpg" width="360" align="left"></img>
<br style="clear:left;"/>

Scrape the results of three hungarian elections - from two very different sources:
- **2018**: html tables, taken apart with BeautifulSoup
- **2022 and 2026**: the JSON files behind the election office's results site

Let's start with 2018.
- import required libraries

In [ ]:
import requests
from bs4 import BeautifulSoup

- set up basic URIs

In [ ]:
VOTE_BASE = 'https://static.valasztas.hu/dyn/pv18/szavossz/hu/'
OVERALL = 'oevker.html'
BASE_URI = './data/'

- download document

In [ ]:
vote_response = requests.get(VOTE_BASE + OVERALL)
print(vote_response.status_code)

- extract data with beautifulsoup

In [ ]:
vote_soup = BeautifulSoup(vote_response.content, "html.parser") 
containers = vote_soup.find('table', {'border': '1'}).find_all('tr')
print(len(containers))
containers[:5]

- get the items out of the tablerows

In [ ]:
rows = [row.find_all('td') for row in containers]
rows[:5]

- "transform" the data into a table-like format

In [ ]:
for row in rows[:5]:
    print([r.getText() for r in row])

- for our analysis, we need the region, the subregion and the links

**Watch out.** The page's html is malformed: the very first `<tr>` of the table
*wraps all the other rows*, so `find_all('td')` on it returns all **636** cells at
once. Take it at face value and you get 107 constituencies instead of 106, with
the first one counted twice - and seven duplicated candidates in the final csv.

A real data row has exactly 6 cells, so check the shape of the row before you
trust it.

In [ ]:
REGIONS = []
for row in rows:
    if len(row) != 6:      # skip the malformed wrapper row
        continue
    REGIONS.append([row[0].getText(), row[2].getText(), row[1].find('a').get('href')])
REGIONS[:5]

In [ ]:
print('Number of regions:', len(REGIONS))

# there are exactly 106 single-member constituencies in Hungary
assert len(REGIONS) == 106, len(REGIONS)
assert len({href for _, _, href in REGIONS}) == 106

- get the detailed information for each region

In [ ]:
results = []

for city, region, sub_url in REGIONS:
    print("Downloading and processing data for {} - {} ...".format(city, region), end='')
    region_response = requests.get(VOTE_BASE + sub_url)
    region_soup = BeautifulSoup(region_response.content, "html.parser")
    region_container = (region_soup
                        .find(string='A szavazatok száma jelöltenként')
                        .find_next('table')
                        .find_all('tr'))
    region_rows = [row.find_all('td') for row in region_container][1:] # remove empty header
    # every candidate will go to a new row
    for row in region_rows:
        results.append([city, region] + [r.getText() for r in row][:-1]) # remove the last 'tick column'
    print("Done.")

- let's look at the detailed information

In [ ]:
print(results[:5])
print('-' * 79)
print('Number of candidates:', len(results))

- transform the items

In [ ]:
cleaned_results =[]

for row in results:
    cleaned_results.append(
        [item.replace(u'\xa0', u'').replace(u'%', u'').strip() # replace the unneeded characters
         for item in row]
    )
cleaned_results[:5]    

Now we can finally save it!

In [ ]:
import csv

import pandas as pd

header = ['region', 'subregion', 'subid', 'name', 'party', 'votes', 'votes %']
filename = './data/vote2018.csv'

# `sep` and `quoting` are chosen to reproduce the file that is already in the
# repo byte for byte, so re-running this notebook does not churn it.
(pd.DataFrame(cleaned_results, columns=header)
 .to_csv(filename, index=False, sep=';', quoting=csv.QUOTE_ALL))

----

# 2022 and 2026: the election API

2018 was html tables. For 2022 and 2026 there is nothing to parse and nothing to clean: the results come as **JSON**.

The results sites ([2022](https://vtr.valasztas.hu/ogy2022), [2026](https://vtr.valasztas.hu/ogy2026)) are javascript applications: download them with `requests` and all you get is a loading spinner. But the data they show has to come from *somewhere*, and that somewhere is exactly what the Network tab shows you (chapter IX, section III). For 2026 it starts with

```
https://vtr.valasztas.hu/ogy2026/data/config.json
```

a three-line file that tells you which **snapshot** of the data to read. No API key, no cookies, no user-agent games.

Downloading a JSON file is something we will do again and again, so `helpers.py` has a function for it: `download_json`.

In [ ]:
import pandas as pd

from helpers import download_json, download_election_results

BASE_2026 = 'https://vtr.valasztas.hu/ogy2026/data/'

config = download_json(BASE_2026 + 'config.json')
config

Every result file hangs off that version number. We need four of them:

| file | what is in it |
|---|---|
| `ver/OevkAdatok.json` | the 106 constituencies and their names |
| `ver/EgyeniJeloltek.json` | every candidate: name, district, party |
| `ver/Szervezetek.json` | the nominating organisations and their full names |
| `szavossz/OevkJkv.json` | the vote counts, per candidate, per district |

The first three come from the `ver` (register) snapshot, the last one from `szavossz` (the count). Let's look into the count:

In [ ]:
count = download_json(BASE_2026 + config['szavossz'] + '/szavossz/OevkJkv.json')['list']
print(len(count), 'districts')

# the first district: one item for every candidate who ran there
count[0]['egyeni_jkv']['tetelek'][:3]

Each item is one candidate: `ej_id` (which candidate), `szavazat` (how many votes) and - for the winner only - `mandatum` (the seat).

To turn this into a table, we look up the candidate's name in the register by `ej_id`, the district's name by its codes, and the party by *its* code. Parties running together have more than one code, so we look each one up and join the names with `'-'`; independents have no party at all.

That is a lot of looking up - and it is exactly the same for both elections, because the election office used the same API in 2022 and in 2026. So it lives in `helpers.py`, as `download_election_results(year)`. Open `helpers.py` and read it: it is the same idea as the 2018 scraping above, with dictionaries instead of html.

In [ ]:
help(download_election_results)

Now both elections, with the same few lines. Check the data before trusting it: Hungary has 106 single-member constituencies, so there must be 106 districts, with exactly one winner in each.

In [ ]:
for year in [2022, 2026]:
    rows = download_election_results(year)
    votes = (pd.DataFrame(rows)
             .sort_values(['region', 'subregion', 'votes'], ascending=[True, True, False])
             .reset_index(drop=True))

    print(year, ':', len(votes), 'candidates in', votes['subregion'].nunique(), 'districts')
    assert votes['subregion'].nunique() == 106
    assert (votes.groupby('subregion')['winner'].sum() == 1).all()

    votes.to_csv('./data/vote' + str(year) + '.csv', index=False)

Who won the districts in 2026 (the last year of the loop)?

In [ ]:
votes.loc[votes['winner'], 'party'].value_counts()

Both files have the same columns, so everything in `python101_10.ipynb` works on either of them - try it with `vote2022.csv`!

In [ ]:
pd.read_csv('./data/vote2022.csv').head()